In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("inc_flag","0")

In [0]:
inc_flag = dbutils.widgets.get("inc_flag")
print(inc_flag)


In [0]:
df_src = spark.sql('''
select distinct(Model_ID) as Model_ID, model_category from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales`
''')
df_src.display()

In [0]:
if spark.catalog.tableExists('cars_catalog.gols.dim_model') :
    df_sink = spark.sql('''
                        select dim_model_key,Model_ID, model_category 
                        from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales' 
                        ''')

else :
    df_sink = spark.sql('''select 1 as dim_model_key,Model_ID, model_category from parquet.`abfss://silver@storageaccountjan2026.dfs.core.windows.net/carsales` where 1=0 ''')




In [0]:
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink,df_src.Model_ID == df_sink.Model_ID,'left').select(df_src.Model_ID,df_src.model_category,df_sink.dim_model_key)
df_filter.display()

In [0]:
df_filter_old = df_filter.filter(df_filter.dim_model_key.isNotNull())
df_filter_old.display()


In [0]:
df_filter_new = df_filter.filter(df_filter.dim_model_key.isNull()).select("Model_ID","model_category")
df_filter_new.display()